In [10]:
import os
import json
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

In [3]:
# STEP 1: 서로 다른 json 파일에 저장된 데이터를 하나의 데이터프레임으로 결합

def load_and_prepare_data(normal_path, phishing_path):
    # 지정된 폴더에서 일반/보이스 피싱 json 파일들을 읽어와
    # 'text' 키에 문자열 데이터를 저장하고, 'label' 키에 0 또는 1의 레이블을 저장합니다.

    data_list = []
    # 1. 통화 데이터 처리(label = 0)
    for filename in os.listdir(normal_path):
    # normal_path 폴더에서 모든 파일 목록을 가져옴
        if filename.endswith(".json"):
            file_path = os.path.join(normal_path, filename)

            with open(file_path, 'r', encoding= 'utf-8') as f:
                json_data = json.load(f)
            # 파일을 열고 json 데이터를 읽음

            dialogs = json_data['dataSet']['dialogs']
            # 일반 대화내용은 구조가 복잡하므로 대화만 순서대로 추출
            conversation_parts = []
            # 각 대화 내용을 한 문장으로 이어붙이기 위한 리스트
            for dialog in dialogs:
                conversation_parts.append(dialog['text'])

            full_conversation = " ".join(conversation_parts)
            # 모든 대화 내용을 하나의 문장으로 결합
  
            data_list.append({'text': full_conversation, 'label': 0})

    # 2. 보이스 피싱 데이터 처리(label = 1)
    # phishing_path 폴더에서 모든 파일 목록을 가져옴
    for filename in os.listdir(phishing_path):
        if filename.endswith(".json"):
            file_path = os.path.join(phishing_path, filename)
            with open(file_path, 'r', encoding= 'utf-8') as f:
                json_data = json.load(f)
            full_conversation = json_data['text']
            # 보이스피싱 json은 'text'키에 전체 대화가 들어있음
            data_list.append({'text': full_conversation, 'label': 1})
        
        # 3. 파이썬 리스트를 pandas DataFrame으로 변환
        df = pd.DataFrame(data_list)
        return df
        # return data_list


In [12]:
def load_and_prepare_data(normal_path, phishing_path):
    """
    [함수 기능]
    검증 데이터 폴더(val_data)에서 모든 json 파일을 읽어와,
    'text'와 실제 정답인 'label'을 가진 표(DataFrame)로 만들어 반환합니다.
    
    [파라미터 설명]
    - normal_path (str): 일반 통화 json 파일들이 들어있는 폴더 경로
    - phishing_path (str): 보이스피싱 json 파일들이 들어있는 폴더 경로
    
    [사용 목적]
    학습 때 사용했던 데이터 로딩 로직과 동일한 방식으로, 
    모델 성능 평가에 사용할 데이터를 준비하기 위해 작성되었습니다.
    """
    data_list = []

    # 1. 일반 통화 데이터 처리 (정답 label=0)
    for filename in os.listdir(normal_path):
        if filename.endswith(".json"):
            file_path = os.path.join(normal_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            dialogs = json_data['dataSet']['dialogs']
            conversation_parts = [dialog['text'] for dialog in dialogs]
            full_conversation = " ".join(conversation_parts)
            data_list.append({'text': full_conversation, 'label': 0})

    # 2. 보이스피싱 데이터 처리 (정답 label=1)
    for filename in os.listdir(phishing_path):
        if filename.endswith(".json"):
            file_path = os.path.join(phishing_path, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)

            full_conversation = json_data['text']
            data_list.append({'text': full_conversation, 'label': 1})
    df = pd.DataFrame(data_list)
    return df

In [13]:
# 데이터 폴더 경로 설정

normal_data_dir = "./data/normal"
phishing_data_dir = "./data/phishing"

df = load_and_prepare_data(normal_data_dir, phishing_data_dir)
# 위에서 만든 함수를 호출하여 데이터 프레임 형성
pd.set_option('display.max_rows', None)
print(len(df))

810


In [14]:
# STEP 2: 모델 학습 준비

model_name = "monologg/koelectra-base-v3-discriminator"
# 사용할 모델이름

tokenizer = AutoTokenizer.from_pretrained(model_name)
# 1. 토크나이저 불러오기(글자를 숫자로)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# 2. 모델불러오기(분류 문제용)
# num_label=2 : 이진 분류 문제이므로 2개의 레이블 사용

train_dataset = Dataset.from_pandas(df)
# pandas dataframe을 huggingface가 사용하는 dataset으로 변환


# 토큰화 함수
def tokenize_function(examples):
    return tokenizer(
        examples["text"],  # 토큰화할 문장
        padding="max_length",  # 패딩의 사이즈에 맞춰서 패딩
        truncation=True,  # 문장이 패딩 사이즈 넘으면 잘라라
    )


tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
# 전체 데이터셋에 토큰화 함수 적용
print("토큰화 완료")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/810 [00:00<?, ? examples/s]

토큰화 완료


In [15]:
# STEP 3: 모델 학습 실행

# Huggingface 라이브러리의 표준적인 설정 방식.

training_args = TrainingArguments(
    output_dir="./results",  # 학습 결과물 저장할 폴더
    num_train_epochs=3,  # 전체 데이터 3번 반복해서 학습
    per_device_train_batch_size=2,  # 한번에 학습할 데이터 개수
    logging_dir="./logs",  # 학습 로그 저장할 폴더
    logging_steps=10,  # 10번 학습할 때마다 로그 출력
    report_to="tensorboard",  # 학습 로그를 tensorboard에 저장
)

# Trainer 객체 생성(모델, 설정, 학습 데이터를 하나로 묶어주는 역할)
trainer = Trainer(
    model=model, args=training_args, train_dataset=tokenized_train_dataset
)

# 모델학습

print("모델 학습 시작")
trainer.train()
print("모델 학습 완료")

# STEP 4: 모델 저장

save_directory = "./fine-tuned-phishing-model"
print(f" 모델 저장 경로: {save_directory}")
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print("모델 저장 완료")

모델 학습 시작


Step,Training Loss
10,0.695500
20,0.689300
30,0.583900
40,0.305100
50,0.141400
60,0.046200
70,0.017900
80,0.011100
90,0.007400
100,0.005400


모델 학습 완료
 모델 저장 경로: ./fine-tuned-phishing-model
모델 저장 완료
